In [5]:
 # ============================================================
# GK-02 BLACK BOX — ROUND 4 RECONSTRUCTION
# Team BB-003
# ============================================================

import os
import numpy as np
import pandas as pd
import joblib

from google.colab import files

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    accuracy_score,
    confusion_matrix
)

print("=" * 60)
print("GK-02 ROUND 4 RECONSTRUCTION")
print("Team BB-003")
print("=" * 60)

# ------------------------------------------------------------
# 1. LOAD THE EXACT ROUND-4 DATASET
# ------------------------------------------------------------

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "nuthanreddy12/blackbox-ai-participant-template/"
    "main/round-4/gk02_query_export_latest150.csv"
)

DATA_FILE = "gk02_query_export_latest150.csv"

try:
    print("\nLoading bundled GK-02 dataset...")
    df = pd.read_csv(DATA_URL)
    print("Loaded dataset from GitHub.")

except Exception:
    print("\nDataset not found online.")
    print("Please upload gk02_query_export_latest150.csv")
    uploaded = files.upload()

    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

# ------------------------------------------------------------
# 2. VALIDATE SCHEMA
# ------------------------------------------------------------

required = [
    "score",
    "decision",
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("\nSchema validation: PASSED")

# ------------------------------------------------------------
# 3. FEATURES
# ------------------------------------------------------------

features = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "ward",
    "years_registered"
]

numeric_features = [
    "age",
    "baseline_score",
    "comorbidity_ratio",
    "dependants",
    "prior_visits",
    "recent_admissions",
    "requested_beds",
    "vitals_index",
    "years_registered"
]

categorical_features = ["ward"]

X = df[features].copy()

y = df["score"].astype(float)

y_decision = (
    df["decision"]
    .astype(str)
    .str.upper()
    .eq("APPROVE")
    .astype(int)
)

# ------------------------------------------------------------
# 4. PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "polynomial",
        PolynomialFeatures(
            degree=2,
            include_bias=False
        )
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

# ------------------------------------------------------------
# 5. RIDGE POLYNOMIAL SURROGATE
# ------------------------------------------------------------

ridge = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "regressor",
        Ridge(alpha=2.0)
    )
])

# ------------------------------------------------------------
# 6. 5-FOLD OUT-OF-FOLD VALIDATION
# ------------------------------------------------------------

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = cross_val_predict(
    ridge,
    X,
    y,
    cv=kf
)

oof = np.clip(
    oof,
    0,
    1
)

r2 = r2_score(
    y,
    oof
)

mae = mean_absolute_error(
    y,
    oof
)

# ------------------------------------------------------------
# 7. OPTIMIZE DECISION THRESHOLD
# ------------------------------------------------------------

best_accuracy = -1.0
best_threshold = None

for threshold in np.arange(
    0.4400,
    0.4801,
    0.0001
):

    predicted = (
        oof >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_decision,
        predicted
    )

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = float(threshold)

predicted_decisions = (
    oof >= best_threshold
).astype(int)

cm = confusion_matrix(
    y_decision,
    predicted_decisions,
    labels=[0, 1]
)

# ------------------------------------------------------------
# 8. FINAL METRICS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL ROUND 4 METRICS")
print("=" * 60)

print(
    f"R²                         : {r2:.6f}"
)

print(
    f"MAE                        : {mae:.6f}"
)

print(
    f"Decision Accuracy (0-1)    : {best_accuracy:.4f}"
)

print(
    f"Decision Accuracy (%)      : {best_accuracy * 100:.2f}%"
)

print(
    f"Correct / Total            : "
    f"{int(round(best_accuracy * len(y)))}/{len(y)}"
)

print(
    f"Wrong                       : "
    f"{len(y) - int(round(best_accuracy * len(y)))}"
)

print(
    f"Optimized threshold        : "
    f"{best_threshold:.4f}"
)

print("\nConfusion Matrix")
print("[DECLINE, APPROVE]")
print(cm)

# ------------------------------------------------------------
# 9. TRAIN FINAL MODEL ON ALL DATA
# ------------------------------------------------------------

ridge.fit(
    X,
    y
)

# ------------------------------------------------------------
# 10. SAVE MODEL
# ------------------------------------------------------------

package = {
    "model": ridge,
    "threshold": best_threshold,
    "features": features,
    "decision_accuracy_cv": float(best_accuracy),
    "r2_cv": float(r2),
    "mae_cv": float(mae)
}

joblib.dump(
    package,
    "GK02_Round4_Reconstruction.pkl"
)

print(
    "\nSaved model: "
    "GK02_Round4_Reconstruction.pkl"
)

# ------------------------------------------------------------
# 11. SAMPLE PREDICTION
# ------------------------------------------------------------

sample = X.iloc[[0]]

actual_score = float(
    y.iloc[0]
)

predicted_score = float(
    np.clip(
        ridge.predict(sample)[0],
        0,
        1
    )
)

predicted_decision = (
    "APPROVE"
    if predicted_score >= best_threshold
    else "DECLINE"
)

print("\n" + "=" * 60)
print("SAMPLE PREDICTION")
print("=" * 60)

print(
    f"Actual Score       : "
    f"{actual_score:.4f}"
)

print(
    f"Predicted Score    : "
    f"{predicted_score:.4f}"
)

print(
    f"Absolute Error     : "
    f"{abs(actual_score - predicted_score):.4f}"
)

print(
    f"Predicted Decision : "
    f"{predicted_decision}"
)

# ------------------------------------------------------------
# 12. NEW INPUT HELPER
# ------------------------------------------------------------

def predict_gk02(row_dict):
    row = pd.DataFrame([row_dict])[features]

    score = float(
        np.clip(
            ridge.predict(row)[0],
            0,
            1
        )
    )

    decision = (
        "APPROVE"
        if score >= best_threshold
        else "DECLINE"
    )

    return {
        "score": score,
        "decision": decision
    }

print("\nFunction ready:")
print("predict_gk02({...})")

GK-02 ROUND 4 RECONSTRUCTION
Team BB-003

Loading bundled GK-02 dataset...

Dataset not found online.
Please upload gk02_query_export_latest150.csv


Saving gk02_query_export_latest150.csv to gk02_query_export_latest150 (1).csv
Rows    : 150
Columns : 13

Schema validation: PASSED

FINAL ROUND 4 METRICS
R²                         : 0.998662
MAE                        : 0.001804
Decision Accuracy (0-1)    : 0.9933
Decision Accuracy (%)      : 99.33%
Correct / Total            : 149/150
Wrong                       : 1
Optimized threshold        : 0.4589

Confusion Matrix
[DECLINE, APPROVE]
[[  4   1]
 [  0 145]]

Saved model: GK02_Round4_Reconstruction.pkl

SAMPLE PREDICTION
Actual Score       : 0.9789
Predicted Score    : 0.9776
Absolute Error     : 0.0013
Predicted Decision : APPROVE

Function ready:
predict_gk02({...})
